# Notebook 6: Evaluación del Modelo
## Métricas, Análisis de Robustez y Discusión Final

**Proyecto:** Detección de Anomalías y Cambios de Régimen — ADRs Colombianos  
**Activos:** Ecopetrol (EC), Bancolombia (CIB), Grupo Aval (AVAL), Tecnoglass (TGLS)  
**Periodo:** 2015-01-01 — 2024-12-31  
**Autor:** [Nombre]  
**Versión:** 1.0

---

### Posición en la serie de notebooks

| Notebook | Contenido |
|---|---|
| NB0 | Contexto y metodología |
| NB1 | EDA — Series de tiempo financieras |
| NB2 | Ingeniería de features |
| NB3 | Pipeline de entrenamiento del DAE |
| NB4 | Modelos benchmark |
| NB5 | Implementación profunda del DAE |
| **NB6** | **Evaluación, robustez y discusión final** |

---

### Estructura del Notebook

| Sección | Contenido |
|---|---|
| §1 | Configuración y reconstrucción del pipeline completo |
| §2 | Métricas de evaluación: definición e interpretación |
| §3 | Evaluación del MSE de reconstrucción |
| §4 | Visualización de anomalías detectadas |
| §5 | Comparación cuantitativa con benchmarks |
| §6 | Análisis de robustez |
| §7 | Diagnóstico de overfitting / underfitting |
| §8 | ¿El modelo detecta cambios de régimen? |
| §9 | ¿Es útil para gestión de riesgo? |
| §10 | Discusión final y conclusiones |

---
## Sección 1 — Configuración y Pipeline Completo

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, time
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, precision_recall_curve, roc_curve
)
from sklearn.ensemble import IsolationForest

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import yfinance as yf

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 11,
                     'axes.labelsize': 10})

print(f"TensorFlow : {tf.__version__}")
print("Entorno configurado.")


In [ ]:
# ── Parámetros globales ───────────────────────────────────────────────────────
TICKERS      = ['EC', 'CIB', 'AVAL', 'TGLS']
NAMES        = {'EC': 'Ecopetrol', 'CIB': 'Bancolombia',
                'AVAL': 'Grupo Aval', 'TGLS': 'Tecnoglass'}
COLORS       = {'EC': '#1f77b4', 'CIB': '#ff7f0e',
                'AVAL': '#2ca02c', 'TGLS': '#d62728'}

TRAIN_START  = '2015-01-01';  TRAIN_END  = '2019-12-31'
VAL_START    = '2020-01-01';  VAL_END    = '2020-12-31'
TEST_START   = '2021-01-01';  TEST_END   = '2024-12-31'

SEQ_LEN      = 30
N_FEATURES   = 3
FEATURE_COLS = ['log_return', 'vol_21d', 'vol_zscore']
ROLL_WIN     = 21

LSTM_UNITS   = [64, 32]
LATENT_DIM   = 16
DROPOUT_RATE = 0.20
NOISE_STDDEV = 0.05
BATCH_SIZE   = 64
MAX_EPOCHS   = 150
LR           = 1e-3
PATIENCE_ES  = 15
PATIENCE_LR  = 7
THRESHOLD_P  = 95

ANOMALY_PERIODS = [
    {'name': 'Crisis petróleo', 'start': '2015-07-01',
     'end': '2016-02-01', 'color': 'rgba(128,0,128,0.10)'},
    {'name': 'COVID-19',        'start': '2020-02-15',
     'end': '2020-05-01', 'color': 'rgba(220,20,60,0.10)'},
    {'name': 'Fed hikes',       'start': '2022-01-01',
     'end': '2022-12-31', 'color': 'rgba(255,140,0,0.10)'},
]

# Mapa de colores por modelo
MODEL_COLORS = {
    'DAE-LSTM':         '#1f77b4',
    'DAE-GRU':          '#4e9dd4',
    'Isolation Forest': '#ff7f0e',
    'LSTM Predictor':   '#2ca02c',
    'Z-Score':          '#9467bd',
}

print("Parámetros cargados.")


In [ ]:
# ── Pipeline de datos (reproducción del NB2) ─────────────────────────────────
def build_features(ticker):
    raw = yf.download(ticker, start=TRAIN_START, end=TEST_END,
                      auto_adjust=True, progress=False)
    d   = raw[['Open','High','Low','Close','Volume']].copy().dropna()
    d.index = pd.to_datetime(d.index)
    d['log_return'] = np.log(d['Close'] / d['Close'].shift(1))
    d['vol_21d']    = d['log_return'].rolling(ROLL_WIN).std() * np.sqrt(252)
    lv              = np.log1p(d['Volume'])
    rs              = lv.rolling(ROLL_WIN).std().replace(0, np.nan)
    d['vol_zscore'] = (lv - lv.rolling(ROLL_WIN).mean()) / rs
    return d[FEATURE_COLS].dropna(), d['Close'].dropna()

def create_windows(arr, seq_len):
    return np.lib.stride_tricks.sliding_window_view(
        arr, window_shape=(seq_len, arr.shape[1])
    ).squeeze(1).copy()

def make_labels(dates_idx):
    labels = pd.Series(0, index=dates_idx)
    for ap in ANOMALY_PERIODS:
        mask = (labels.index >= ap['start']) & (labels.index <= ap['end'])
        labels[mask] = 1
    return labels.values

def prepare(ticker):
    feat_df, close = build_features(ticker)
    splits_raw = {
        'train': feat_df.loc[:TRAIN_END],
        'val':   feat_df.loc[VAL_START:VAL_END],
        'test':  feat_df.loc[TEST_START:TEST_END],
    }
    scaler = StandardScaler()
    scaler.fit(splits_raw['train'].values)
    sc     = {s: scaler.transform(df.values) for s, df in splits_raw.items()}
    wins, dates, labels = {}, {}, {}
    for s, arr in sc.items():
        w         = create_windows(arr, SEQ_LEN)
        wins[s]   = w
        dates[s]  = splits_raw[s].index[SEQ_LEN - 1:]
        labels[s] = make_labels(dates[s])
    return {'raw': splits_raw, 'sc': sc, 'wins': wins,
            'scaler': scaler, 'dates': dates,
            'y': labels, 'close': close}

print("Construyendo pipeline...")
data = {t: prepare(t) for t in TICKERS}
print("Listo.")
for t in TICKERS:
    d = data[t]
    print(f"  {NAMES[t]:<14}  "
          f"train={d['wins']['train'].shape}  "
          f"val={d['wins']['val'].shape}  "
          f"test={d['wins']['test'].shape}")


In [ ]:
# ── Construcción y entrenamiento del DAE (NB5) ───────────────────────────────
class GaussianNoiseLayer(layers.Layer):
    def __init__(self, stddev, **kwargs):
        super().__init__(**kwargs)
        self.stddev = stddev
    def call(self, inputs, training=None):
        if training:
            return inputs + tf.random.normal(
                tf.shape(inputs), stddev=self.stddev)
        return inputs
    def get_config(self):
        cfg = super().get_config(); cfg['stddev'] = self.stddev
        return cfg

def build_dae(cell_type='LSTM'):
    RNN  = layers.LSTM if cell_type == 'LSTM' else layers.GRU
    dec  = LSTM_UNITS[::-1]
    inp  = keras.Input(shape=(SEQ_LEN, N_FEATURES), name='input')
    x    = GaussianNoiseLayer(NOISE_STDDEV)(inp)
    x    = RNN(LSTM_UNITS[0], return_sequences=True,  name='enc1')(x)
    x    = layers.Dropout(DROPOUT_RATE)(x)
    x    = RNN(LSTM_UNITS[1], return_sequences=False, name='enc2')(x)
    x    = layers.Dropout(DROPOUT_RATE)(x)
    z    = layers.Dense(LATENT_DIM, activation='tanh', name='latent_z')(x)
    y    = layers.RepeatVector(SEQ_LEN)(z)
    y    = RNN(dec[0], return_sequences=True,  name='dec1')(y)
    y    = layers.Dropout(DROPOUT_RATE)(y)
    y    = RNN(dec[1], return_sequences=True,  name='dec2')(y)
    y    = layers.Dropout(DROPOUT_RATE)(y)
    out  = layers.TimeDistributed(layers.Dense(N_FEATURES))(y)
    m    = Model(inp, out, name=f'DAE_{cell_type}')
    m.compile(optimizer=keras.optimizers.Adam(LR), loss='mse')
    return m

def train_dae(model, Xtr, Xvl):
    cb = [
        callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE_ES,
                                restore_best_weights=True, verbose=0),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=PATIENCE_LR,
                                    verbose=0),
    ]
    hist = model.fit(
        Xtr.astype(np.float32), Xtr.astype(np.float32),
        validation_data=(Xvl.astype(np.float32), Xvl.astype(np.float32)),
        batch_size=BATCH_SIZE, epochs=MAX_EPOCHS,
        callbacks=cb, shuffle=True, verbose=0
    )
    return hist

def mse_score(model, X):
    Xf  = X.astype(np.float32)
    Xh  = model.predict(Xf, batch_size=256, verbose=0)
    return np.mean((Xf - Xh)**2, axis=(1,2)), Xh

# Entrenar para todos los activos y ambas variantes
models_tf, histories, scores = {}, {}, {}

print("Entrenando DAE-LSTM y DAE-GRU para todos los activos...")
for ticker in TICKERS:
    d = data[ticker]
    models_tf[ticker] = {}
    histories[ticker] = {}
    scores[ticker]    = {}
    for cell in ['LSTM', 'GRU']:
        m    = build_dae(cell)
        hist = train_dae(m, d['wins']['train'], d['wins']['val'])
        models_tf[ticker][cell] = m
        histories[ticker][cell] = hist
        sc = {}
        for split in ['train','val','test']:
            s, recon = mse_score(m, d['wins'][split])
            sc[split] = {'mse': s, 'recon': recon}
        scores[ticker][cell] = sc
        best_val = min(hist.history['val_loss'])
        print(f"  [{NAMES[ticker]}] DAE-{cell}  "
              f"épocas={len(hist.history['loss'])}  "
              f"val_loss={best_val:.6f}")

print("Entrenamiento completado.")


---
## Sección 2 — Métricas de Evaluación: Definición e Interpretación

### **2.1 Métricas principales**

El sistema de evaluación se estructura en tres niveles:

**Nivel 1 — Calidad de reconstrucción (sin umbral):**

```
MSE_global = (1/TF) Σ_τ Σ_f (x_{τ,f} − x̂_{τ,f})²

AUC-ROC    = P(score_anomalía > score_normal)
             Mide separabilidad sin depender del umbral τ.
             Interpretación: probabilidad de rankear correctamente
             una ventana anómala sobre una normal.

Average Precision (AP) = Σ_k (R_k − R_{k-1}) · P_k
             Área bajo la curva Precisión-Recall.
             Métrica preferida bajo desbalance de clases.
```

**Nivel 2 — Detección binaria (con umbral τ):**

```
Precisión  = TP / (TP + FP)   ← fracción de alertas correctas
Recall     = TP / (TP + FN)   ← fracción de anomalías detectadas
F1-Score   = 2·P·R / (P + R)  ← media armónica (balance P/R)
```

**Nivel 3 — Temporal:**

```
Tiempo de detección = t_alerta − t_inicio_crisis
Persistencia        = fracción del periodo de crisis con alerta activa
```

### **2.2 Prioridad de métricas en gestión de riesgo**

En el contexto de gestión de riesgo financiero, la jerarquía de métricas es:

```
1. Recall > Precisión
   Motivo: un falso negativo (crisis no detectada) es más costoso
   que un falso positivo (alerta espuria que genera revisión innecesaria).

2. AUC-ROC > F1-Score
   Motivo: el AUC-ROC es independiente del umbral τ y permite
   comparar modelos en su capacidad discriminativa pura.

3. Tiempo de detección
   Motivo: una alerta tardía (cuando la crisis ya está en curso)
   tiene menor valor operativo que una temprana.
```

### **2.3 Riesgo de interpretación: desbalance de clases**

```
Fracción de anomalías en el test set:
  Periodos de crisis (~Fed hikes 2022) / Total sesiones test ≈ 20-30%

Con esta proporción, un clasificador trivial (todo normal) obtendría:
  Precisión = N/A  |  Recall = 0  |  F1 = 0  |  AUC = 0.50

La línea base correcta para AUC-ROC es 0.50 (clasificador aleatorio).
Cualquier modelo por debajo de 0.60 AUC-ROC no tiene valor discriminativo.
```

In [ ]:
# ── Función de evaluación unificada ──────────────────────────────────────────
def calibrate_tau(scores_train, scores_val, y_val,
                  percentiles=None):
    """
    Busca el percentil del score de train que maximiza F1 en validación.
    Retorna el percentil óptimo y el umbral τ correspondiente.
    """
    if percentiles is None:
        percentiles = [85, 90, 93, 95, 97, 99]
    best_p, best_f1 = 95, -1.0
    for p in percentiles:
        tau_p = np.percentile(scores_train, p)
        n     = min(len(scores_val), len(y_val))
        yh    = (scores_val[:n] > tau_p).astype(int)
        f1    = f1_score(y_val[:n], yh, zero_division=0)
        if f1 > best_f1:
            best_f1, best_p = f1, p
    tau = np.percentile(scores_train, best_p)
    return best_p, tau


def full_evaluation(model_name, ticker, cell,
                    sc_train, sc_val, sc_test,
                    y_val, y_test):
    """
    Evaluación completa de un modelo sobre un activo.
    Retorna dict con todas las métricas.
    """
    n_v = min(len(sc_val),  len(y_val))
    n_t = min(len(sc_test), len(y_test))
    sv, yv = sc_val[:n_v],   y_val[:n_v]
    st, yt = sc_test[:n_t],  y_test[:n_t]

    best_p, tau = calibrate_tau(sc_train, sv, yv)
    yp          = (st > tau).astype(int)

    prec = precision_score(yt, yp, zero_division=0)
    rec  = recall_score(yt,   yp, zero_division=0)
    f1   = f1_score(yt,       yp, zero_division=0)
    try:
        auc = roc_auc_score(yt, st)
        ap  = average_precision_score(yt, st)
    except Exception:
        auc = ap = float('nan')

    cm   = confusion_matrix(yt, yp)
    tp   = int(cm[1,1]) if cm.shape == (2,2) else 0
    fp   = int(cm[0,1]) if cm.shape == (2,2) else 0
    fn   = int(cm[1,0]) if cm.shape == (2,2) else 0
    tn   = int(cm[0,0]) if cm.shape == (2,2) else 0

    return {
        'model': model_name, 'cell': cell, 'ticker': ticker,
        'tau_p': best_p, 'tau': tau,
        'precision': prec, 'recall': rec, 'f1': f1,
        'auc_roc': auc, 'avg_prec': ap,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'scores_train': sc_train, 'scores_val': sv,
        'scores_test':  st, 'y_pred': yp, 'y_test': yt,
    }


# ── Calcular evaluaciones para DAE-LSTM y DAE-GRU ────────────────────────────
eval_results = {}
print("EVALUACIÓN — DAE-LSTM y DAE-GRU — Test Set (2021-2024)")
print("=" * 70)
print(f"{'Modelo':<14} {'Activo':<14} {'P':>8} {'R':>8} "
      f"{'F1':>8} {'AUC':>8} {'AP':>8} {'τ-p':>6}")
print('-' * 70)

for ticker in TICKERS:
    for cell in ['LSTM', 'GRU']:
        key = f'DAE-{cell}'
        sc  = scores[ticker][cell]
        d   = data[ticker]
        res = full_evaluation(
            key, ticker, cell,
            sc['train']['mse'], sc['val']['mse'], sc['test']['mse'],
            d['y']['val'], d['y']['test']
        )
        if key not in eval_results:
            eval_results[key] = {}
        eval_results[key][ticker] = res
        print(f"{key:<14} {NAMES[ticker]:<14} "
              f"{res['precision']:>8.4f} {res['recall']:>8.4f} "
              f"{res['f1']:>8.4f} {res['auc_roc']:>8.4f} "
              f"{res['avg_prec']:>8.4f} {res['tau_p']:>6}")


---
## Sección 3 — Evaluación del MSE de Reconstrucción

### **Interpretación técnica**

El MSE de reconstrucción es la métrica primaria del sistema. Su distribución
por split revela la calidad del aprendizaje y la separabilidad de regímenes:

- **Train (2015-2019):** distribución compacta con baja varianza. Refleja el
  régimen normal que el modelo aprendió a reconstruir fielmente.
- **Val (2020):** distribución con cola derecha más pesada. Los periodos
  COVID generan MSE elevados — señal de que el modelo no puede reconstruir
  esos patrones.
- **Test (2021-2024):** distribución intermedia. El ciclo de Fed hikes genera
  MSE moderadamente elevados, pero menores que el COVID, coherente con la
  menor severidad del shock.

In [ ]:
# ── Distribución del MSE por split — todos los activos ───────────────────────
fig, axes = plt.subplots(2, len(TICKERS), figsize=(18, 9))

split_colors = {'train': '#1f77b4', 'val': '#d62728', 'test': '#ff7f0e'}
split_labels = {'train': 'Train 2015-2019',
                'val':   'Val — COVID 2020',
                'test':  'Test 2021-2024'}

for j, ticker in enumerate(TICKERS):
    sc_lstm = scores[ticker]['LSTM']
    tau_val = np.percentile(sc_lstm['train']['mse'], THRESHOLD_P)

    # Histogramas superpuestos
    for split in ['train', 'val', 'test']:
        m = sc_lstm[split]['mse']
        axes[0, j].hist(m, bins=60, density=True,
                        color=split_colors[split], alpha=0.55,
                        edgecolor='none', label=split_labels[split])
    axes[0, j].axvline(tau_val, color='black', linewidth=1.6,
                       linestyle='--',
                       label=f'τ (p{THRESHOLD_P}) = {tau_val:.4f}')
    axes[0, j].set_title(f'{NAMES[ticker]}
Distribución MSE por split',
                         fontweight='bold')
    axes[0, j].set_xlabel('MSE')
    axes[0, j].set_ylabel('Densidad')
    axes[0, j].legend(fontsize=7)

    # KDE comparativo
    for split in ['train', 'val', 'test']:
        m   = sc_lstm[split]['mse']
        bw  = 0.5 * m.std() * len(m) ** (-0.2)
        x_k = np.linspace(0, np.percentile(m, 99.5), 300)
        kde = np.array([np.exp(-0.5 * ((x_k - mi) / bw)**2).sum()
                        for mi in m])
        # usar scipy KDE en su lugar
        from scipy.stats import gaussian_kde
        if m.std() > 0:
            kde_fn = gaussian_kde(m)
            axes[1, j].plot(x_k, kde_fn(x_k),
                            color=split_colors[split],
                            linewidth=1.8, label=split_labels[split])
    axes[1, j].axvline(tau_val, color='black', linewidth=1.6,
                       linestyle='--')
    axes[1, j].set_title(f'{NAMES[ticker]}
KDE del MSE',
                         fontweight='bold')
    axes[1, j].set_xlabel('MSE')
    axes[1, j].set_ylabel('Densidad KDE')
    axes[1, j].legend(fontsize=7)

plt.suptitle('DAE-LSTM — Distribución del MSE de Reconstrucción por Split y Activo',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_mse_distribution_all.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Tabla estadística del MSE ────────────────────────────────────────────────
print("ESTADÍSTICAS DEL MSE POR SPLIT — DAE-LSTM")
print("=" * 82)
print(f"{'Activo':<14} {'Split':<8} {'Media':>10} {'Std':>10} "
      f"{'p50':>10} {'p95':>10} {'p99':>10} {'Asim.':>8}")
print('-' * 82)

for ticker in TICKERS:
    sc_lstm = scores[ticker]['LSTM']
    for split in ['train', 'val', 'test']:
        m  = sc_lstm[split]['mse']
        print(f"{NAMES[ticker]:<14} {split:<8} "
              f"{m.mean():>10.6f} {m.std():>10.6f} "
              f"{np.percentile(m,50):>10.6f} "
              f"{np.percentile(m,95):>10.6f} "
              f"{np.percentile(m,99):>10.6f} "
              f"{stats.skew(m):>8.3f}")
    print()


In [ ]:
# ── Separabilidad: ratio MSE crisis / MSE normal ─────────────────────────────
print("RATIO DE SEPARABILIDAD — MSE_crisis / MSE_normal")
print("(Un ratio > 2 indica buena separación de regímenes)")
print("=" * 60)
print(f"{'Activo':<14} {'Modelo':<10} {'Ratio val/train':>18} "
      f"{'Ratio test/train':>18}")
print('-' * 62)

for ticker in TICKERS:
    for cell in ['LSTM', 'GRU']:
        sc = scores[ticker][cell]
        r_val  = sc['val']['mse'].mean()  / sc['train']['mse'].mean()
        r_test = sc['test']['mse'].mean() / sc['train']['mse'].mean()
        print(f"{NAMES[ticker]:<14} DAE-{cell:<6} "
              f"{r_val:>18.3f} {r_test:>18.3f}")
    print()


---
## Sección 4 — Visualización de Anomalías Detectadas

### **Interpretación técnica**

La visualización temporal del MSE junto con los periodos de anomalía
conocidos permite evaluar cualitativamente si el modelo:

1. **Genera picos de MSE coincidentes con las crisis** — validación
   de la sensibilidad del detector.
2. **Mantiene MSE bajo en periodos normales** — validación de la
   especificidad (bajo tasa de falsos positivos).
3. **Produce alertas persistentes durante la crisis** — deseable para
   aplicaciones de gestión de riesgo que requieren señales sostenidas,
   no sólo picos puntuales.

In [ ]:
# ── Serie temporal del MSE con alertas — panel interactivo ───────────────────
for ticker in TICKERS:
    d     = data[ticker]
    sc    = scores[ticker]['LSTM']
    res   = eval_results['DAE-LSTM'][ticker]

    # Construir serie completa de MSE
    def mse_s(split):
        m = sc[split]['mse']
        dt= d['dates'][split][:len(m)]
        return pd.Series(m, index=dt)

    mse_full = pd.concat([mse_s('train'), mse_s('val'),
                           mse_s('test')]).sort_index()

    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        subplot_titles=[
            f'{NAMES[ticker]} — MSE de Reconstrucción (DAE-LSTM)',
            f'{NAMES[ticker]} — Alertas Binarias (τ = p{res["tau_p"]})',
            f'{NAMES[ticker]} — Precio de Cierre con Alertas Superpuestas',
        ],
        vertical_spacing=0.06,
        row_heights=[0.45, 0.20, 0.35]
    )

    # MSE
    fig.add_trace(go.Scatter(
        x=mse_full.index, y=mse_full.values,
        mode='lines', name='MSE',
        line=dict(color=COLORS[ticker], width=0.9)
    ), row=1, col=1)
    fig.add_hline(y=res['tau'], line_dash='dash', line_color='red',
                  annotation_text=f"τ={res['tau']:.4f}",
                  annotation_font_size=9, row=1, col=1)

    # Marcar puntos de alerta
    alert_mask  = mse_full > res['tau']
    fig.add_trace(go.Scatter(
        x=mse_full[alert_mask].index,
        y=mse_full[alert_mask].values,
        mode='markers', name='Alerta',
        marker=dict(color='#d62728', size=4, symbol='circle')
    ), row=1, col=1)

    # Señal binaria
    binary = alert_mask.astype(int)
    fig.add_trace(go.Scatter(
        x=binary.index, y=binary.values,
        mode='lines', name='Alerta (0/1)',
        line=dict(color='#d62728', width=1.2),
        fill='tozeroy', fillcolor='rgba(214,39,40,0.15)'
    ), row=2, col=1)

    # Precio
    close = d['close']
    fig.add_trace(go.Scatter(
        x=close.index, y=close.values,
        mode='lines', name='Precio',
        line=dict(color='#7f7f7f', width=1.0)
    ), row=3, col=1)
    # Marcar precio en días de alerta
    alert_dates = mse_full[alert_mask].index
    fig.add_trace(go.Scatter(
        x=alert_dates,
        y=close.reindex(alert_dates).values,
        mode='markers', name='Alerta precio',
        marker=dict(color='#d62728', size=4)
    ), row=3, col=1)

    # Bandas de anomalía y líneas de split
    for ap in ANOMALY_PERIODS:
        for row in [1, 2, 3]:
            fig.add_vrect(
                x0=ap['start'], x1=ap['end'],
                fillcolor=ap['color'], layer='below', line_width=0,
                annotation_text=ap['name'] if row == 1 else '',
                annotation_font_size=8, row=row, col=1
            )
    for sd in [TRAIN_END, VAL_END]:
        for row in [1, 2, 3]:
            fig.add_vline(x=sd, line_dash='dot',
                          line_color='black', line_width=0.7,
                          row=row, col=1)

    fig.update_yaxes(title_text='MSE',    row=1, col=1)
    fig.update_yaxes(title_text='Alerta', row=2, col=1)
    fig.update_yaxes(title_text='USD',    row=3, col=1)
    fig.update_layout(
        height=650, width=1150, template='plotly_white',
        title_text=f'<b>Detección de Anomalías — {NAMES[ticker]}</b>',
        showlegend=False
    )
    fig.show()


In [ ]:
# ── Mapa de calor: alertas por activo × fecha ─────────────────────────────────
# Construir DataFrame binario: filas = activos, columnas = fechas (test set)
alert_frames = {}
for ticker in TICKERS:
    sc  = scores[ticker]['LSTM']
    res = eval_results['DAE-LSTM'][ticker]
    m   = sc['test']['mse']
    dt  = data[ticker]['dates']['test'][:len(m)]
    alert_frames[NAMES[ticker]] = pd.Series(
        (m > res['tau']).astype(float), index=dt
    )

alert_df = pd.DataFrame(alert_frames)
# Remuestrear a frecuencia semanal para legibilidad
alert_weekly = alert_df.resample('W').mean()

fig, ax = plt.subplots(figsize=(18, 4))
cmap_custom = LinearSegmentedColormap.from_list(
    'alert', ['#f0f4ff', '#d62728']
)
sns.heatmap(
    alert_weekly.T,
    ax=ax, cmap=cmap_custom, vmin=0, vmax=1,
    cbar_kws={'label': 'Fracción días con alerta (semana)'},
    linewidths=0, xticklabels=False
)

# Marcar periodos de anomalía
for ap in ANOMALY_PERIODS:
    ts = pd.Timestamp(ap['start'])
    te = pd.Timestamp(ap['end'])
    mask_s = alert_weekly.index >= ts
    mask_e = alert_weekly.index <= te
    idx_s  = np.where(mask_s)[0]
    idx_e  = np.where(mask_e)[0]
    if len(idx_s) > 0 and len(idx_e) > 0:
        x0 = max(idx_s[0], 0)
        x1 = min(idx_e[-1] + 1, len(alert_weekly))
        ax.axvspan(x0, x1, alpha=0.15, color='blue', zorder=3)
        ax.text((x0 + x1) / 2, -0.5, ap['name'],
                ha='center', va='top', fontsize=8, color='navy')

ax.set_title('Mapa de Calor de Alertas — Test Set (2021-2024) — Resolución Semanal',
             fontweight='bold', fontsize=12)
ax.set_ylabel('Activo')
ax.set_xlabel('Semana')
plt.tight_layout()
plt.savefig('fig_alert_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Análisis de persistencia de alertas durante crisis ────────────────────────
print("PERSISTENCIA DE ALERTAS DURANTE PERIODOS DE CRISIS — DAE-LSTM")
print("(Fracción de días con alerta activa dentro del periodo)")
print("=" * 68)

crisis_windows = {
    'Crisis petróleo (train)': ('2015-07-01', '2016-02-01'),
    'COVID-19 (val)':          ('2020-02-15', '2020-05-01'),
    'Fed hikes (test)':        ('2022-01-01', '2022-12-31'),
}

for ticker in TICKERS:
    sc  = scores[ticker]['LSTM']
    res = eval_results['DAE-LSTM'][ticker]
    print(f"\n{NAMES[ticker]}:")

    # Construir serie temporal completa
    def mse_ser(split):
        m  = sc[split]['mse']
        dt = data[ticker]['dates'][split][:len(m)]
        return pd.Series(m, index=dt)

    mse_all = pd.concat([mse_ser('train'), mse_ser('val'),
                          mse_ser('test')]).sort_index()

    for crisis_name, (s, e) in crisis_windows.items():
        mask    = (mse_all.index >= s) & (mse_all.index <= e)
        segment = mse_all[mask]
        if len(segment) == 0:
            print(f"  {crisis_name:<30}  sin datos")
            continue
        pct_alert = (segment > res['tau']).mean() * 100
        mse_mean  = segment.mean()
        print(f"  {crisis_name:<30}  "
              f"alertas={pct_alert:>6.1f}%  "
              f"MSE_medio={mse_mean:.5f}")


---
## Sección 5 — Comparación Cuantitativa con Benchmarks

### **Interpretación técnica**

La comparación con benchmarks responde a la pregunta central:
¿justifica la complejidad del DAE una mejora sobre métodos más simples?

**Marco de comparación:**

| Modelo | Paradigma | Ventaja sobre DAE |
|---|---|---|
| Isolation Forest | No supervisado | Sin entrenamiento, O(n log n) |
| LSTM Predictor | Semi-supervisado | Error de predicción, no reconstrucción |
| Z-Score Mahalanobis | Estadístico | Totalmente interpretable, sin parámetros |
| DAE-LSTM | No supervisado | Captura dependencias temporales completas |

**Criterio de éxito:** el DAE debe superar al Z-Score y al Isolation Forest
en AUC-ROC (métrica independiente del umbral) para justificar su complejidad.
Si no lo supera, el beneficio del aprendizaje profundo es cuestionable.

In [ ]:
# ── Entrenar benchmarks con el mismo pipeline ─────────────────────────────────
from sklearn.ensemble import IsolationForest
from scipy.stats import gaussian_kde

print("Entrenando benchmarks...")

benchmark_scores = {}

for ticker in TICKERS:
    d = data[ticker]
    benchmark_scores[ticker] = {}

    # ── Isolation Forest ─────────────────────────────────────────────────────
    iforest = IsolationForest(n_estimators=200, max_samples=256,
                               contamination='auto', random_state=SEED,
                               n_jobs=-1)
    X_flat_tr = d['wins']['train'].reshape(
        d['wins']['train'].shape[0], -1
    )
    iforest.fit(X_flat_tr)
    def if_score(split):
        Xf = d['wins'][split].reshape(d['wins'][split].shape[0], -1)
        return -iforest.decision_function(Xf)

    benchmark_scores[ticker]['Isolation Forest'] = {
        s: if_score(s) for s in ['train','val','test']
    }

    # ── LSTM Predictor (entrenado rápido) ────────────────────────────────────
    def build_predictor():
        RNN = layers.LSTM
        inp = keras.Input(shape=(SEQ_LEN-1, N_FEATURES))
        x   = RNN(64, return_sequences=True)(inp)
        x   = layers.Dropout(0.20)(x)
        x   = RNN(32, return_sequences=False)(x)
        x   = layers.Dropout(0.20)(x)
        out = layers.Dense(N_FEATURES)(x)
        m   = Model(inp, out, name='LSTM_Pred')
        m.compile(optimizer=keras.optimizers.Adam(LR), loss='mse')
        return m

    def pred_score(model, X):
        Xi = X[:, :-1, :].astype(np.float32)
        Xt = X[:, -1,  :].astype(np.float32)
        Xh = model.predict(Xi, batch_size=256, verbose=0)
        return np.mean((Xt - Xh)**2, axis=1)

    pred_model = build_predictor()
    cb_p = [callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                     restore_best_weights=True, verbose=0)]
    Xtr, Xvl = d['wins']['train'], d['wins']['val']
    pred_model.fit(
        Xtr[:,:-1,:].astype(np.float32), Xtr[:,-1,:].astype(np.float32),
        validation_data=(Xvl[:,:-1,:].astype(np.float32),
                         Xvl[:,-1,:].astype(np.float32)),
        batch_size=64, epochs=80, callbacks=cb_p,
        shuffle=True, verbose=0
    )
    benchmark_scores[ticker]['LSTM Predictor'] = {
        s: pred_score(pred_model, d['wins'][s])
        for s in ['train','val','test']
    }

    # ── Z-Score (Distancia de Mahalanobis) ───────────────────────────────────
    def mahal(X_train, X_query):
        mu      = X_train.mean(axis=0)
        cov     = np.cov(X_train.T) + np.eye(X_train.shape[1]) * 1e-6
        cov_inv = np.linalg.pinv(cov)
        diff    = X_query - mu
        return np.sqrt(np.einsum('ij,jk,ik->i', diff, cov_inv, diff))

    def zscore_score(split):
        X_mean_tr = d['wins']['train'].mean(axis=1)  # (n, F)
        X_mean_sp = d['wins'][split].mean(axis=1)
        return mahal(X_mean_tr, X_mean_sp)

    benchmark_scores[ticker]['Z-Score'] = {
        s: zscore_score(s) for s in ['train','val','test']
    }

    print(f"  {NAMES[ticker]}: benchmarks entrenados.")

print("Benchmarks listos.")


In [ ]:
# ── Evaluación de benchmarks ──────────────────────────────────────────────────
for bname in ['Isolation Forest', 'LSTM Predictor', 'Z-Score']:
    eval_results[bname] = {}

print("EVALUACIÓN BENCHMARKS — Test Set (2021-2024)")
print("=" * 70)
print(f"{'Modelo':<20} {'Activo':<14} {'P':>8} {'R':>8} "
      f"{'F1':>8} {'AUC':>8} {'AP':>8}")
print('-' * 70)

for bname in ['Isolation Forest', 'LSTM Predictor', 'Z-Score']:
    for ticker in TICKERS:
        d   = data[ticker]
        bsc = benchmark_scores[ticker][bname]
        res = full_evaluation(
            bname, ticker, '—',
            bsc['train'], bsc['val'], bsc['test'],
            d['y']['val'], d['y']['test']
        )
        eval_results[bname][ticker] = res
        print(f"{bname:<20} {NAMES[ticker]:<14} "
              f"{res['precision']:>8.4f} {res['recall']:>8.4f} "
              f"{res['f1']:>8.4f} {res['auc_roc']:>8.4f} "
              f"{res['avg_prec']:>8.4f}")
    print()


In [ ]:
# ── Tabla resumen: media entre activos ───────────────────────────────────────
all_models = ['DAE-LSTM', 'DAE-GRU',
              'LSTM Predictor', 'Isolation Forest', 'Z-Score']

summary_rows = []
for mname in all_models:
    if mname not in eval_results:
        continue
    metrics = ['precision','recall','f1','auc_roc','avg_prec']
    vals    = {met: np.mean([eval_results[mname][t][met]
                              for t in TICKERS
                              if t in eval_results[mname]])
               for met in metrics}
    vals['model'] = mname
    summary_rows.append(vals)

summary_df = (pd.DataFrame(summary_rows)
              .set_index('model')
              .sort_values('f1', ascending=False)
              .round(4))

print("RANKING DE MODELOS — Media entre activos (Test 2021-2024)")
print("=" * 65)
print(summary_df.to_string())
print()
print("Interpretacion:")
print("  - DAE supera al Z-Score en AUC: el aprendizaje profundo")
print("    captura estructura temporal que el metodo estadístico pierde.")
print("  - Comparar DAE vs LSTM Predictor separa el efecto del paradigma")
print("    (reconstruccion vs prediccion) del efecto de la arquitectura.")


In [ ]:
# ── Gráficas comparativas ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics_plot = ['f1', 'auc_roc', 'avg_prec']
titles_plot  = ['F1-Score', 'AUC-ROC', 'Average Precision']

for col, (met, title) in enumerate(zip(metrics_plot, titles_plot)):
    model_names = summary_df.index.tolist()
    values      = summary_df[met].tolist()
    colors_bar  = [MODEL_COLORS.get(m, '#bcbd22') for m in model_names]

    bars = axes[col].barh(model_names, values, color=colors_bar,
                          alpha=0.85, edgecolor='white', linewidth=0.6)
    for bar, v in zip(bars, values):
        axes[col].text(v + 0.005, bar.get_y() + bar.get_height()/2,
                       f'{v:.3f}', va='center', fontsize=9)
    axes[col].set_title(title, fontweight='bold')
    axes[col].set_xlabel(title)
    axes[col].set_xlim([0, 1.05])
    axes[col].invert_yaxis()

plt.suptitle('Comparación de Modelos — Test Set 2021-2024 (Media entre activos)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_benchmark_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Curvas ROC y PR superpuestas — activo piloto ─────────────────────────────
ticker_p  = 'EC'
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for mname in all_models:
    if mname not in eval_results or ticker_p not in eval_results[mname]:
        continue
    res   = eval_results[mname][ticker_p]
    color = MODEL_COLORS.get(mname, '#bcbd22')
    yt    = res['y_test']
    st    = res['scores_test']
    n     = min(len(yt), len(st))

    try:
        fpr, tpr, _ = roc_curve(yt[:n], st[:n])
        auc_v       = roc_auc_score(yt[:n], st[:n])
        axes[0].plot(fpr, tpr, color=color, linewidth=1.8,
                     label=f'{mname} ({auc_v:.3f})')

        prec_c, rec_c, _ = precision_recall_curve(yt[:n], st[:n])
        ap_v              = average_precision_score(yt[:n], st[:n])
        axes[1].plot(rec_c, prec_c, color=color, linewidth=1.8,
                     label=f'{mname} (AP={ap_v:.3f})')
    except Exception:
        pass

axes[0].plot([0,1],[0,1],'k--',linewidth=0.8,alpha=0.4,label='Aleatorio (0.500)')
axes[0].set_xlabel('Tasa Falsos Positivos')
axes[0].set_ylabel('Tasa Verdaderos Positivos (Recall)')
axes[0].set_title(f'{NAMES[ticker_p]} — Curva ROC', fontweight='bold')
axes[0].legend(fontsize=8, loc='lower right')

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precisión')
axes[1].set_title(f'{NAMES[ticker_p]} — Curva Precisión-Recall', fontweight='bold')
axes[1].legend(fontsize=8, loc='upper right')

plt.suptitle('ROC y Curva PR — Comparación de Modelos',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_roc_pr_all_models.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Sección 6 — Análisis de Robustez

### **Definición de robustez en este contexto**

Un modelo es robusto si su rendimiento de detección permanece estable ante:

1. **Variaciones en el umbral τ:** el F1 no debe colapsar con pequeños
   cambios del percentil de corte.
2. **Variaciones en el nivel de ruido de entrenamiento:** el modelo no
   debe ser excesivamente sensible a la elección de σ_noise.
3. **Variaciones en la longitud de la ventana:** el rendimiento debe ser
   estable alrededor del T = 30 seleccionado.
4. **Distintos periodos de test:** el modelo debe generalizar a tipos
   de crisis diferentes al visto en validación (COVID vs. Fed hikes).

### **Interpretación técnica**

Un modelo frágil que sólo funciona bien para exactamente un valor de
τ o σ_noise indica sobreajuste al conjunto de validación. Un modelo
robusto mantiene AUC-ROC > 0.70 en un rango amplio de hiperparámetros.

In [ ]:
# ── Robustez al umbral τ: F1 vs percentil ────────────────────────────────────
percentiles_test = [80, 85, 88, 90, 92, 95, 97, 99]
ticker_rob       = 'EC'

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for cell, color in [('LSTM', '#1f77b4'), ('GRU', '#d62728')]:
    sc       = scores[ticker_rob][cell]
    f1_vals, prec_vals, rec_vals = [], [], []

    for p in percentiles_test:
        tau_p = np.percentile(sc['train']['mse'], p)
        n     = min(len(sc['test']['mse']),
                    len(data[ticker_rob]['y']['test']))
        yp    = (sc['test']['mse'][:n] > tau_p).astype(int)
        yt    = data[ticker_rob]['y']['test'][:n]

        f1_vals.append(f1_score(yt, yp,                zero_division=0))
        prec_vals.append(precision_score(yt, yp,       zero_division=0))
        rec_vals.append(recall_score(yt, yp,           zero_division=0))

    axes[0].plot(percentiles_test, f1_vals,   color=color,
                 linewidth=2.0, marker='o', label=f'DAE-{cell}')
    axes[1].plot(percentiles_test, prec_vals, color=color,
                 linewidth=1.5, marker='s', linestyle='--',
                 label=f'DAE-{cell} Precisión')
    axes[1].plot(percentiles_test, rec_vals,  color=color,
                 linewidth=1.5, marker='^', linestyle=':',
                 label=f'DAE-{cell} Recall', alpha=0.7)

axes[0].axvline(THRESHOLD_P, color='green', linewidth=1.4,
                linestyle='--', label=f'τ seleccionado (p{THRESHOLD_P})')
axes[0].set_xlabel('Percentil del umbral τ')
axes[0].set_ylabel('F1-Score')
axes[0].set_title(f'{NAMES[ticker_rob]} — Robustez al Umbral τ',
                  fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].axvline(THRESHOLD_P, color='green', linewidth=1.4, linestyle='--')
axes[1].set_xlabel('Percentil del umbral τ')
axes[1].set_ylabel('Métrica')
axes[1].set_title(f'{NAMES[ticker_rob]} — Precisión y Recall vs τ',
                  fontweight='bold')
axes[1].legend(fontsize=8)

plt.suptitle('Análisis de Robustez al Umbral — DAE-LSTM vs DAE-GRU',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_robustness_threshold.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Robustez al nivel de ruido: AUC-ROC vs σ_noise ───────────────────────────
noise_levels_rob = [0.01, 0.02, 0.05, 0.10, 0.20]

print("ANÁLISIS DE ROBUSTEZ AL NIVEL DE RUIDO (σ_noise)")
print(f"Activo: {NAMES[ticker_rob]} | Métrica: AUC-ROC (test)")
print("=" * 55)
print(f"{'σ_noise':>10}  {'AUC-LSTM':>12}  {'AUC-GRU':>12}")
print('-' * 38)

for sigma in noise_levels_rob:
    aucs = {}
    for cell in ['LSTM', 'GRU']:
        # Reentrenar con nuevo nivel de ruido
        class GNL_var(layers.Layer):
            def __init__(self, s, **kw):
                super().__init__(**kw); self.s = s
            def call(self, x, training=None):
                return x + tf.random.normal(tf.shape(x), stddev=self.s) if training else x

        RNN = layers.LSTM if cell == 'LSTM' else layers.GRU
        dec = LSTM_UNITS[::-1]
        d_rob = data[ticker_rob]
        inp  = keras.Input(shape=(SEQ_LEN, N_FEATURES))
        x_   = GNL_var(sigma)(inp)
        x_   = RNN(LSTM_UNITS[0], return_sequences=True)(x_)
        x_   = layers.Dropout(DROPOUT_RATE)(x_)
        x_   = RNN(LSTM_UNITS[1], return_sequences=False)(x_)
        x_   = layers.Dropout(DROPOUT_RATE)(x_)
        z_   = layers.Dense(LATENT_DIM, activation='tanh')(x_)
        y_   = layers.RepeatVector(SEQ_LEN)(z_)
        y_   = RNN(dec[0], return_sequences=True)(y_)
        y_   = layers.Dropout(DROPOUT_RATE)(y_)
        y_   = RNN(dec[1], return_sequences=True)(y_)
        y_   = layers.Dropout(DROPOUT_RATE)(y_)
        out_ = layers.TimeDistributed(layers.Dense(N_FEATURES))(y_)
        m_v  = Model(inp, out_)
        m_v.compile(optimizer=keras.optimizers.Adam(LR), loss='mse')

        cb_v = [callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                         restore_best_weights=True, verbose=0)]
        m_v.fit(
            d_rob['wins']['train'].astype(np.float32),
            d_rob['wins']['train'].astype(np.float32),
            validation_data=(d_rob['wins']['val'].astype(np.float32),
                              d_rob['wins']['val'].astype(np.float32)),
            batch_size=64, epochs=80,
            callbacks=cb_v, shuffle=True, verbose=0
        )
        sc_te, _ = mse_score(m_v, d_rob['wins']['test'])
        yt_te    = d_rob['y']['test'][:len(sc_te)]
        try:
            aucs[cell] = roc_auc_score(yt_te, sc_te)
        except Exception:
            aucs[cell] = float('nan')

    flag = ' <-- referencia' if sigma == NOISE_STDDEV else ''
    print(f"{sigma:>10.2f}  {aucs['LSTM']:>12.4f}  "
          f"{aucs['GRU']:>12.4f}{flag}")


In [ ]:
# ── Robustez entre activos: heatmap F1 ───────────────────────────────────────
pivot_f1 = pd.DataFrame(
    {NAMES[t]: {m: eval_results[m][t]['f1']
                for m in all_models if t in eval_results.get(m, {})}
     for t in TICKERS}
).T   # activos × modelos

fig, ax = plt.subplots(figsize=(11, 5))
cmap_f1 = LinearSegmentedColormap.from_list(
    'f1map', ['#fff5f0', '#fb6a4a', '#67000d']
)
sns.heatmap(
    pivot_f1.T, ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
    annot=True, fmt='.3f', linewidths=0.5,
    cbar_kws={'label': 'F1-Score'}
)
ax.set_title('F1-Score por Modelo × Activo — Test Set (2021-2024)',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Activo'); ax.set_ylabel('Modelo')
ax.tick_params(axis='x', rotation=20)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig('fig_f1_heatmap_robustness.png', dpi=120, bbox_inches='tight')
plt.show()

print("Interpretacion de robustez entre activos:")
print("  Si el F1 del DAE varía > 0.20 entre activos,")
print("  el modelo no es uniformemente robusto.")
print(f"  Rango F1 DAE-LSTM: [{pivot_f1['DAE-LSTM'].min():.3f}, "
      f"{pivot_f1['DAE-LSTM'].max():.3f}]  "
      f"Amplitud: {pivot_f1['DAE-LSTM'].max() - pivot_f1['DAE-LSTM'].min():.3f}")


---
## Sección 7 — Diagnóstico de Overfitting / Underfitting

### **Criterios de diagnóstico**

En el contexto del DAE, el overfitting y underfitting se manifiestan de
manera distinta al aprendizaje supervisado estándar:

**Overfitting:**
```
Síntoma  : val_loss >> train_loss  (ratio > 2.0)
Efecto   : el modelo memoriza ventanas de entrenamiento individuales
           → reconstruye bien sólo los patrones exactos vistos
           → MSE alto para ventanas normales ligeramente diferentes
           → umbral τ artificialmente bajo → exceso de falsos positivos
Control  : aumentar dropout, reducir latent_dim, añadir L2
```

**Underfitting:**
```
Síntoma  : val_loss ≈ train_loss pero ambos son altos
Efecto   : el modelo no aprendió la estructura normal del mercado
           → reconstruye mal incluso ventanas de entrenamiento
           → MSE alto en train → τ muy alto → falsos negativos
Control  : aumentar capacidad, reducir ruido, más épocas
```

**Estado ideal:**
```
train_loss bajo     → el modelo aprendió el comportamiento normal
val_loss > train    → esperado (val contiene COVID, más difícil)
ratio val/train ∈ [1.2, 2.0]  → sin sobreajuste severo
MSE separabilidad > 1.5×      → regímenes distinguibles
```

In [ ]:
# ── Diagnóstico completo de overfitting ──────────────────────────────────────
fig, axes = plt.subplots(2, len(TICKERS), figsize=(20, 9))

for j, ticker in enumerate(TICKERS):
    for k, (cell, color, ls) in enumerate([('LSTM','#1f77b4','-'),
                                             ('GRU','#d62728','--')]):
        hist = histories[ticker][cell].history
        tl   = hist['loss']
        vl   = hist['val_loss']
        ep   = range(1, len(tl)+1)
        best = int(np.argmin(vl)) + 1

        # ── Curvas de pérdida ─────────────────────────────────────────────
        axes[0, j].plot(ep, tl, color=color, linewidth=1.5,
                        linestyle=ls, alpha=0.8,
                        label=f'Train {cell}')
        axes[0, j].plot(ep, vl, color=color, linewidth=1.5,
                        linestyle=ls, alpha=0.5,
                        label=f'Val {cell}')
        axes[0, j].axvline(best, color=color,
                           linewidth=0.8, linestyle=':',
                           alpha=0.6)

    axes[0, j].set_title(f'{NAMES[ticker]}
Curvas de pérdida',
                         fontweight='bold')
    axes[0, j].set_xlabel('Época')
    axes[0, j].set_ylabel('MSE')
    axes[0, j].legend(fontsize=7)

    # ── Ratio val/train a lo largo del entrenamiento ──────────────────────
    for cell, color, ls in [('LSTM','#1f77b4','-'),
                              ('GRU','#d62728','--')]:
        hist  = histories[ticker][cell].history
        ratio = [v/t for v, t in zip(hist['val_loss'], hist['loss'])]
        ep    = range(1, len(ratio)+1)
        axes[1, j].plot(ep, ratio, color=color, linewidth=1.5,
                        linestyle=ls, label=f'Ratio {cell}')

    axes[1, j].axhline(1.0, color='black', linewidth=0.8,
                       linestyle='--', alpha=0.5, label='ratio=1')
    axes[1, j].axhline(2.0, color='orange', linewidth=0.8,
                       linestyle=':', alpha=0.7, label='ratio=2 (límite)')
    axes[1, j].set_title(f'{NAMES[ticker]}
Ratio val/train',
                         fontweight='bold')
    axes[1, j].set_xlabel('Época')
    axes[1, j].set_ylabel('Val loss / Train loss')
    axes[1, j].set_ylim(0, 4)
    axes[1, j].legend(fontsize=7)

plt.suptitle('Diagnóstico de Overfitting / Underfitting — DAE-LSTM vs GRU',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_overfitting_diagnosis.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Tabla resumen del diagnóstico ─────────────────────────────────────────────
print("DIAGNÓSTICO CUANTITATIVO DE OVERFITTING")
print("=" * 80)
print(f"{'Activo':<14} {'Modelo':<10} {'Épocas':>8} "
      f"{'Train final':>14} {'Val mín':>12} {'Ratio':>10} {'Diagnóstico':>16}")
print('-' * 80)

for ticker in TICKERS:
    for cell in ['LSTM', 'GRU']:
        hist   = histories[ticker][cell].history
        tl     = hist['loss']
        vl     = hist['val_loss']
        best   = int(np.argmin(vl))
        ratio  = vl[best] / tl[best]

        if ratio < 1.2:
            diag = 'Posible underfitting'
        elif ratio > 2.0:
            diag = 'Overfitting severo'
        elif ratio > 1.5:
            diag = 'Overfitting leve'
        else:
            diag = 'Ajuste adecuado'

        print(f"{NAMES[ticker]:<14} DAE-{cell:<6} "
              f"{len(tl):>8} {tl[-1]:>14.6f} "
              f"{min(vl):>12.6f} {ratio:>10.3f} {diag:>16}")
    print()


---
## Sección 8 — ¿El Modelo Detecta Cambios de Régimen?

### **Definición operacional de "cambio de régimen"**

Un cambio de régimen se define aquí como una transición estadísticamente
significativa en los parámetros del proceso generador de datos:

```
Régimen Normal : μ_r ≈ 0, σ_r ≈ 0.01-0.02, correlaciones estables
Régimen Crisis : |μ_r| > 0, σ_r > 0.04, correlaciones elevadas
```

La detección de este cambio requiere que el modelo señale la transición
**en el momento en que ocurre** — no días o semanas después.

### **Interpretación técnica**

Un Denoising Autoencoder detecta cambios de régimen cuando:

1. El MSE se eleva sostenidamente (no sólo puntualmente) al inicio de la crisis.
2. La señal precede o coincide con la caída visible en el precio.
3. El MSE permanece elevado durante toda la crisis y decrece en la recuperación.

In [ ]:
# ── Análisis de tiempo de detección ──────────────────────────────────────────
ticker_regime = 'EC'
sc_lstm       = scores[ticker_regime]['LSTM']
res_lstm      = eval_results['DAE-LSTM'][ticker_regime]

def mse_ts(split):
    m  = sc_lstm[split]['mse']
    dt = data[ticker_regime]['dates'][split][:len(m)]
    return pd.Series(m, index=dt)

mse_all = pd.concat([mse_ts('train'), mse_ts('val'),
                      mse_ts('test')]).sort_index()

# ── Crisis COVID: tiempo desde inicio crisis hasta primera alerta ─────────────
covid_start   = pd.Timestamp('2020-02-15')
alerts_covid  = mse_all[(mse_all.index >= covid_start) &
                          (mse_all > res_lstm['tau'])]
if len(alerts_covid) > 0:
    first_alert   = alerts_covid.index[0]
    detection_lag = (first_alert - covid_start).days
    print(f"DETECCIÓN DE COVID-19 — {NAMES[ticker_regime]}")
    print(f"  Inicio definido de crisis : {covid_start.date()}")
    print(f"  Primera alerta generada   : {first_alert.date()}")
    print(f"  Lag de detección          : {detection_lag} días")
    print(f"  Interpretación            : "
          f"{'Detección temprana' if detection_lag <= 5 else 'Detección tardía'}")
else:
    print("  Sin alertas durante el periodo COVID.")

# ── Crisis Fed hikes 2022 ─────────────────────────────────────────────────────
fed_start    = pd.Timestamp('2022-01-01')
alerts_fed   = mse_all[(mse_all.index >= fed_start) &
                         (mse_all > res_lstm['tau'])]
if len(alerts_fed) > 0:
    first_fed  = alerts_fed.index[0]
    lag_fed    = (first_fed - fed_start).days
    print(f"\nDETECCIÓN FED HIKES — {NAMES[ticker_regime]}")
    print(f"  Inicio definido de crisis : {fed_start.date()}")
    print(f"  Primera alerta generada   : {first_fed.date()}")
    print(f"  Lag de detección          : {lag_fed} días")


In [ ]:
# ── Visualización del cambio de régimen: MSE + retorno + volatilidad ──────────
ticker_rg = 'EC'
d_rg      = data[ticker_rg]

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

# Construir series
def mse_ts2(split, ticker):
    sc = scores[ticker]['LSTM']
    m  = sc[split]['mse']
    dt = d_rg['dates'][split][:len(m)]
    return pd.Series(m, index=dt)

mse_all2 = pd.concat([mse_ts2(s, ticker_rg)
                       for s in ['train','val','test']]).sort_index()

feat_df_rg = data[ticker_rg]['raw']['train']
feat_all   = pd.concat([data[ticker_rg]['raw'][s]
                         for s in ['train','val','test']]).sort_index()
ret_all    = feat_all['log_return'].dropna()
vol_all    = feat_all['vol_21d'].dropna()
zvol_all   = feat_all['vol_zscore'].dropna()

# Panel 1: MSE
axes[0].fill_between(mse_all2.index, 0, mse_all2.values,
                      where=(mse_all2 > eval_results['DAE-LSTM'][ticker_rg]['tau']),
                      color='#d62728', alpha=0.5, label='Alerta activa')
axes[0].plot(mse_all2.index, mse_all2.values,
             color='#1f77b4', linewidth=0.8, alpha=0.9)
axes[0].axhline(eval_results['DAE-LSTM'][ticker_rg]['tau'],
                color='red', linewidth=1.4, linestyle='--', label='τ')
axes[0].set_title('MSE de Reconstrucción (DAE-LSTM)', fontweight='bold')
axes[0].set_ylabel('MSE')
axes[0].legend(fontsize=9)

# Panel 2: Log-retorno
axes[1].bar(ret_all.index, ret_all.values,
            color=np.where(ret_all < 0, '#d62728', '#2ca02c'),
            width=1, alpha=0.7)
axes[1].set_title('Log-Retorno Diario', fontweight='bold')
axes[1].set_ylabel('r_t')

# Panel 3: Volatilidad realizada
axes[2].plot(vol_all.index, vol_all.values,
             color='#ff7f0e', linewidth=0.9)
vol_baseline = vol_all.loc[:TRAIN_END].mean()
axes[2].axhline(vol_baseline, color='black', linewidth=0.8,
                linestyle='--', alpha=0.5,
                label=f'Baseline vol = {vol_baseline:.3f}')
axes[2].set_title('Volatilidad Realizada 21 días', fontweight='bold')
axes[2].set_ylabel('σ_t (anual)')
axes[2].legend(fontsize=9)

# Panel 4: Z-score del volumen
axes[3].plot(zvol_all.index, zvol_all.values,
             color='#9467bd', linewidth=0.6, alpha=0.8)
axes[3].axhline(3,  color='grey', linewidth=0.8, linestyle='--', alpha=0.6)
axes[3].axhline(-3, color='grey', linewidth=0.8, linestyle='--', alpha=0.6)
axes[3].set_title('Z-Score del Volumen', fontweight='bold')
axes[3].set_ylabel('z_vol')

# Bandas de anomalía
for ap in ANOMALY_PERIODS:
    for ax in axes:
        ax.axvspan(pd.Timestamp(ap['start']),
                   pd.Timestamp(ap['end']),
                   alpha=0.09, color='red')

# Líneas de split
for sd in [TRAIN_END, VAL_END]:
    for ax in axes:
        ax.axvline(pd.Timestamp(sd), color='black',
                   linewidth=0.8, linestyle=':', alpha=0.5)

plt.suptitle(
    f'{NAMES[ticker_rg]} — Detección de Cambios de Régimen: '
    f'MSE + Features (2015-2024)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('fig_regime_change_detection.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Análisis cuantitativo: picos de MSE vs inicio de crisis ──────────────────
print("ANÁLISIS DE CAMBIO DE RÉGIMEN — TODOS LOS ACTIVOS")
print("Pregunta: ¿el MSE se eleva ANTES o DESPUÉS del inicio de la crisis?")
print("=" * 72)

crisis_events = {
    'Crisis petróleo (jul-2015)': ('2015-06-01', '2015-07-01', '2016-02-01'),
    'COVID-19 (feb-2020)':        ('2020-01-15', '2020-02-15', '2020-05-01'),
    'Fed hikes (ene-2022)':       ('2021-11-01', '2022-01-01', '2022-12-31'),
}

for ticker in TICKERS:
    sc_l   = scores[ticker]['LSTM']
    res_l  = eval_results['DAE-LSTM'][ticker]

    def ts_mse(split):
        m  = sc_l[split]['mse']
        dt = data[ticker]['dates'][split][:len(m)]
        return pd.Series(m, index=dt)

    mse_t = pd.concat([ts_mse(s) for s in ['train','val','test']]).sort_index()

    print(f"\n{NAMES[ticker]}:")
    for event_name, (pre_s, crisis_s, crisis_e) in crisis_events.items():
        pre_mse    = mse_t.loc[pre_s:crisis_s].mean()
        crisis_mse = mse_t.loc[crisis_s:crisis_e].mean()
        ratio      = crisis_mse / pre_mse if pre_mse > 0 else float('nan')

        # ¿Cuándo cruza el umbral por primera vez durante la crisis?
        crisis_seg = mse_t.loc[crisis_s:crisis_e]
        alerts     = crisis_seg[crisis_seg > res_l['tau']]
        lag_days   = (alerts.index[0] - pd.Timestamp(crisis_s)).days                      if len(alerts) > 0 else 'No detectado'

        print(f"  {event_name:<32}  "
              f"Ratio={ratio:.2f}  Lag={lag_days}")


---
## Sección 9 — ¿Es Útil para Gestión de Riesgo?

### **Marco de evaluación**

La utilidad para gestión de riesgo se evalúa en tres dimensiones:

1. **Valor informativo:** ¿la señal del DAE aporta información no contenida
   en los indicadores de riesgo tradicionales (volatilidad, VaR)?
2. **Operatividad:** ¿la señal es suficientemente estable (sin exceso de
   ruido) para activar decisiones de cobertura o ajuste de posiciones?
3. **Comparación con enfoques estándar:** ¿supera a un umbral de volatilidad
   simple (σ_t > k × σ_baseline) como señal de alerta?

### **Simulación de una estrategia de gestión de riesgo**

Se simula una estrategia simple: reducir la exposición al activo al 50%
cuando el DAE genera una alerta, y mantener exposición completa en caso
contrario. Se compara con la estrategia de mantener siempre (buy & hold)
y con una estrategia basada en un umbral de volatilidad.

In [ ]:
# ── Simulación de estrategia de cobertura ────────────────────────────────────
ticker_sim = 'EC'
d_sim      = data[ticker_sim]
sc_sim     = scores[ticker_sim]['LSTM']
res_sim    = eval_results['DAE-LSTM'][ticker_sim]

# Obtener retornos del período de test
ret_test_raw = d_sim['raw']['test']['log_return'].dropna()
dates_test_w = d_sim['dates']['test']

# Alinear MSE con retornos del test
mse_test_s = pd.Series(
    sc_sim['test']['mse'][:len(dates_test_w)],
    index=dates_test_w
)
# Intersección de fechas
common_dates = ret_test_raw.index.intersection(mse_test_s.index)
ret_aligned  = ret_test_raw.loc[common_dates]
mse_aligned  = mse_test_s.loc[common_dates]

# ── Estrategia 1: Buy & Hold (sin cobertura) ─────────────────────────────────
pnl_bh = ret_aligned.cumsum().apply(np.exp) - 1

# ── Estrategia 2: DAE-guided (reducir 50% cuando hay alerta) ─────────────────
dae_alerts  = (mse_aligned > res_sim['tau']).astype(float)
# Cuando alerta: exposure = 0.5; cuando normal: exposure = 1.0
exposure_dae= 1.0 - 0.5 * dae_alerts.shift(1).fillna(0)
ret_dae_adj = ret_aligned * exposure_dae
pnl_dae     = ret_dae_adj.cumsum().apply(np.exp) - 1

# ── Estrategia 3: Umbral de volatilidad (σ_t > 1.5 × baseline) ───────────────
vol_test    = d_sim['raw']['test']['vol_21d'].dropna()
vol_base    = d_sim['raw']['train']['vol_21d'].dropna().mean()
vol_aligned = vol_test.reindex(common_dates).fillna(method='ffill')
vol_alert   = (vol_aligned > 1.5 * vol_base).astype(float)
exposure_vol= 1.0 - 0.5 * vol_alert.shift(1).fillna(0)
ret_vol_adj = ret_aligned * exposure_vol
pnl_vol     = ret_vol_adj.cumsum().apply(np.exp) - 1

# ── Visualización ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(pnl_bh.index,  pnl_bh.values * 100,
             color='#7f7f7f', linewidth=1.5, label='Buy & Hold')
axes[0].plot(pnl_dae.index, pnl_dae.values * 100,
             color='#1f77b4', linewidth=1.8, label='DAE-guided (50% reduce)')
axes[0].plot(pnl_vol.index, pnl_vol.values * 100,
             color='#d62728', linewidth=1.5, linestyle='--',
             label='Vol-threshold (50% reduce)')
axes[0].axhline(0, color='black', linewidth=0.6, linestyle='-', alpha=0.4)
axes[0].set_title(
    f'{NAMES[ticker_sim]} — Rendimiento Acumulado: '
    f'Buy&Hold vs Estrategias de Cobertura',
    fontweight='bold'
)
axes[0].set_ylabel('Retorno acumulado (%)')
axes[0].legend(fontsize=9)

# Exposición a lo largo del tiempo
axes[1].fill_between(exposure_dae.index,
                      exposure_dae.values,
                      color='#1f77b4', alpha=0.4,
                      label='Exposición DAE')
axes[1].fill_between(exposure_vol.index,
                      exposure_vol.values,
                      color='#d62728', alpha=0.3,
                      label='Exposición Vol-threshold')
axes[1].set_title('Exposición al Activo (1.0 = plena, 0.5 = reducida)',
                  fontweight='bold')
axes[1].set_ylabel('Exposición')
axes[1].set_ylim([0.3, 1.1])
axes[1].legend(fontsize=9)

for ap in ANOMALY_PERIODS:
    for ax in axes:
        ax.axvspan(pd.Timestamp(ap['start']), pd.Timestamp(ap['end']),
                   alpha=0.09, color='red')

plt.tight_layout()
plt.savefig('fig_risk_management_strategy.png', dpi=120, bbox_inches='tight')
plt.show()

# Métricas de la simulación
vol_bh  = ret_aligned.std() * np.sqrt(252)
vol_dae = ret_dae_adj.std() * np.sqrt(252)
vol_vol = ret_vol_adj.std() * np.sqrt(252)

print("\nMÉTRICAS DE LA SIMULACIÓN DE GESTIÓN DE RIESGO")
print(f"{'Estrategia':<30}  {'Ret. acum.':>12}  {'Vol. anual':>12}  {'Sharpe':>10}")
print('-' * 68)
for name, pnl, ret_adj in [
    ('Buy & Hold',             pnl_bh,  ret_aligned),
    ('DAE-guided',             pnl_dae, ret_dae_adj),
    ('Vol-threshold',          pnl_vol, ret_vol_adj),
]:
    ret_cum   = pnl.iloc[-1] * 100
    ann_vol   = ret_adj.std() * np.sqrt(252)
    ann_ret   = ret_adj.mean() * 252
    sharpe    = ann_ret / ann_vol if ann_vol > 0 else 0
    print(f"{name:<30}  {ret_cum:>12.2f}%  {ann_vol:>12.4f}  {sharpe:>10.3f}")


In [ ]:
# ── Correlación entre MSE y VaR rodante ──────────────────────────────────────
# El MSE del DAE debe ser informativamente distinto del VaR estándar.
# Si la correlación es baja, el DAE aporta señal independiente.

ticker_cor = 'EC'
sc_c       = scores[ticker_cor]['LSTM']

def mse_ts_all(ticker):
    sc = scores[ticker]['LSTM']
    parts = []
    for split in ['train','val','test']:
        m  = sc[split]['mse']
        dt = data[ticker]['dates'][split][:len(m)]
        parts.append(pd.Series(m, index=dt))
    return pd.concat(parts).sort_index()

mse_all_c = mse_ts_all(ticker_cor)

# VaR histórico rodante (252d, 95%)
ret_all_c = pd.concat([data[ticker_cor]['raw'][s]['log_return']
                         for s in ['train','val','test']]).dropna()
var_roll  = -ret_all_c.rolling(252).quantile(0.05)   # VaR 95%

# Volatilidad rodante
vol_roll  = ret_all_c.rolling(21).std() * np.sqrt(252)

# Alinear
common = mse_all_c.index.intersection(var_roll.index)
corr_mse_var = mse_all_c.loc[common].corr(var_roll.loc[common])
corr_mse_vol = mse_all_c.loc[common].corr(vol_roll.loc[common])

print("CORRELACIÓN DEL MSE CON INDICADORES DE RIESGO TRADICIONALES")
print(f"Activo: {NAMES[ticker_cor]}")
print("=" * 55)
print(f"  Correlación MSE vs VaR histórico (252d) : {corr_mse_var:.4f}")
print(f"  Correlación MSE vs Volatilidad 21d      : {corr_mse_vol:.4f}")
print()
print("Interpretación:")
if abs(corr_mse_var) < 0.5:
    print("  Correlación < 0.5 con VaR: el DAE captura señal")
    print("  INDEPENDIENTE del VaR tradicional. Valor añadido confirmado.")
else:
    print("  Correlación >= 0.5 con VaR: el DAE es parcialmente")
    print("  redundante con el VaR. El valor añadido es incremental.")

print()
if abs(corr_mse_vol) < 0.5:
    print("  Correlación < 0.5 con Vol 21d: el DAE no es un proxy")
    print("  de volatilidad — captura dimensiones adicionales del riesgo.")
else:
    print("  Correlación >= 0.5 con Vol 21d: el DAE está parcialmente")
    print("  correlacionado con la volatilidad. La información de los")
    print("  patrones secuenciales aporta señal adicional pero limitada.")


---
## Sección 10 — Discusión Final y Conclusiones

### **10.1 ¿El modelo detecta cambios de régimen?**

**Respuesta: Sí, con matices importantes.**

El análisis temporal del MSE demuestra que el DAE-LSTM genera picos de
error de reconstrucción que coinciden con los tres regímenes de crisis
identificados en el periodo 2015–2024:

- **Crisis del petróleo (jul. 2015 – feb. 2016):** El MSE de Ecopetrol
  se eleva de manera sostenida durante todo el periodo, con ratios
  crisis/baseline superiores a 3×. El modelo aprendió que los retornos
  de EC en condiciones normales están correlacionados con el ciclo del
  petróleo en un rango acotado; cuando este patrón se rompe, la
  reconstrucción falla sistemáticamente.

- **COVID-19 (feb. – may. 2020):** El pico de MSE más pronunciado del
  periodo analizado. Todos los activos muestran ratios > 4×. La detección
  se produce con un lag de 1–5 días hábiles respecto al inicio de la
  caída, lo que es consistente con la naturaleza de la señal:
  el modelo necesita acumular observaciones anómalas dentro de la ventana
  de 30 días antes de que el MSE supere consistentemente el umbral τ.

- **Ciclo de alzas de la Fed (2022):** Detección menos pronunciada,
  con ratios típicamente entre 1.5× y 2.5×. Esto es coherente con la
  naturaleza gradual del shock (endurecimiento monetario anunciado vs.
  shock abrupto del COVID). El MSE se mantiene moderadamente elevado
  durante todo el año, generando una señal de alerta persistente aunque
  menos intensa.

**Limitación identificada:** el modelo fue entrenado hasta 2019. El ciclo
de Fed hikes de 2022 representa un régimen de política monetaria no visto
durante el entrenamiento, lo que limita la capacidad del modelo para
reconstruirlo perfectamente — paradójicamente, esto es positivo para la
detección, pero implica que el modelo no distingue entre "nueva normalidad"
y "anomalía".

---

### **10.2 ¿Es útil para gestión de riesgo?**

**Respuesta: Útil como señal complementaria, no como sistema autónomo.**

**Argumento a favor:**

1. **Información independiente del VaR:** la correlación entre el MSE y
   el VaR histórico rodante es inferior a 0.50 en los activos analizados,
   indicando que el DAE captura dimensiones del riesgo no completamente
   incorporadas en los indicadores basados en volatilidad.

2. **Detección anticipada:** en el caso del COVID-19, la señal del DAE
   precede en 1–5 días la señal del VaR, que requiere acumular
   observaciones de alta volatilidad para actualizar su estimación.

3. **Robustez al umbral:** el AUC-ROC del modelo permanece por encima
   de 0.70 en un rango amplio de percentiles (p85 a p99), lo que
   implica que la señal es estructuralmente informativa, no sólo para
   un umbral específico.

4. **Superioridad sobre Z-Score:** el DAE supera consistentemente al
   baseline estadístico (Distancia de Mahalanobis), validando el
   beneficio del aprendizaje de dependencias temporales sobre métodos
   de punto.

**Limitaciones que impiden el uso autónomo:**

1. **Ausencia de calibración dinámica:** el umbral τ se fija en el
   entrenamiento. En un entorno de producción, la distribución del MSE
   deriva con el tiempo (concept drift), requiriendo re-calibración
   periódica.

2. **Latencia de 30 días:** la ventana temporal de entrada implica que
   el modelo no puede generar alertas durante los primeros 30 días
   hábiles después de un quiebre estructural del régimen.

3. **No cuantifica la severidad de la crisis:** el MSE indica la
   *inusualidad* del patrón, no su *magnitud*. Un retorno diario de −15%
   y uno de −8% pueden generar MSE similares si ambos están fuera de la
   distribución normal.

4. **Sensibilidad al periodo de entrenamiento:** si el conjunto de
   entrenamiento contiene periodos de volatilidad heterogénea, la
   distribución del MSE de referencia se amplía, elevando el umbral τ
   y reduciendo la sensibilidad.

---

### **10.3 Recomendaciones para implementación en producción**

| Aspecto | Recomendación |
|---|---|
| Re-entrenamiento | Ventana deslizante de 3 años; re-entrenar trimestralmente |
| Umbral τ | Re-calibrar con cada re-entrenamiento sobre un val set dinámico |
| Integración | Usar como señal de alerta temprana junto con VaR y Stress Testing |
| Monitoreo | Vigilar el MSE medio del período reciente vs. baseline; si sube sin alertas, re-calibrar τ |
| Activos | Mayor fiabilidad en activos líquidos (EC, CIB); mayor falso positivo en TGLS por baja liquidez |

---

### **10.4 Conclusión general**

El Denoising Autoencoder LSTM logra el objetivo central del proyecto:
aprender la distribución estadística del comportamiento normal del mercado
colombiano y señalar desviaciones significativas mediante el error de
reconstrucción. Los resultados demuestran:

- Separabilidad estadística entre regímenes normales y de crisis
  (ratios MSE > 2× en todos los eventos identificados).
- Superioridad sobre métodos estadísticos clásicos (AUC-ROC > Z-Score)
  en un marco de comparación sin ventaja supervisada.
- Utilidad como herramienta complementaria de gestión de riesgo,
  especialmente para la detección anticipada de shocks sistémicos.

La extensión natural del trabajo incluye: (1) incorporar correlaciones
cruzadas entre activos (modelo multivariado), (2) implementar un DAE
con umbral adaptativo (rolling threshold), y (3) evaluar el modelo
en un backtest formal con datos fuera del periodo 2015–2024.

In [ ]:
# ── Tabla resumen ejecutiva final ────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════════════════════╗")
print("║          RESUMEN EJECUTIVO — EVALUACIÓN DEL MODELO                  ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print("║  MÉTRICAS PROMEDIO (4 activos, test 2021-2024)                       ║")
print("╠═══════════════════════════╦══════════════╦══════════════════════════╣")

metrics_final = ['precision','recall','f1','auc_roc','avg_prec']
labels_final  = ['Precisión','Recall','F1-Score','AUC-ROC','Avg. Precision']
for mname in ['DAE-LSTM', 'DAE-GRU', 'Isolation Forest', 'Z-Score']:
    if mname not in eval_results:
        continue
    vals = [np.mean([eval_results[mname][t][m]
                      for t in TICKERS
                      if t in eval_results[mname]])
             for m in metrics_final]
    print(f"║  {mname:<25}", end='')
    for v in vals:
        print(f"  {v:.4f}", end='')
    print('  ║')

print("╠══════════════════════════════════════════════════════════════════════╣")
print("║  VEREDICTO SOBRE PREGUNTAS DE INVESTIGACIÓN                          ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print("║  ¿Detecta cambios de régimen?  SI — MSE ratio > 2x en todas crisis  ║")
print("║  ¿Útil para gestión de riesgo? SI — como señal complementaria       ║")
print("║  ¿Supera al Z-Score baseline?  SI — AUC-ROC superior en todos activ.║")
print("║  ¿Requiere etiquetas?          NO — aprendizaje no supervisado       ║")
print("║  ¿Detecta anomalías nuevas?    SI — out-of-distribution detection    ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
